# End-to-end cross-layer pipeline

Companion notebook for paper Sec. V-G ("Cross-Layer Data Flow and Compliance Lifecycle"). Runs one full transaction lifecycle through all four ARCA layers using the exact same code the pytest suite and `arca.pipeline.end_to_end_demo` script use — nothing here is notebook-only mock logic.

In [1]:
import sys, pathlib
REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from arca.pipeline.end_to_end_demo import run_demo

outcome = run_demo(verbose=True)

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


STEP 1 — Layer 4: Regulatory Oracle Network -> Layer 1: Jurisdiction Agent
Jurisdiction risk classification — AU: low, SY (FATF grey list): high

STEP 2 — Layer 1 proposes rule updates -> Layer 4 governance validates -> Layer 3 activates
Rule 'jurisdiction-AU-real_estate-transfer' (max_amount=10000000) applied: True
Rule 'jurisdiction-allowlist-real_estate-transfer' applied: True (32 allowed jurisdictions)
Rules 'require-eligibility-real_estate-transfer', 'max-risk-score-real_estate-transfer' and 'block-anomaly-real_estate-transfer' also applied.

STEP 3 — Layer 4: Asset Registry
Registered asset ASSET-AU-001 (real_estate, jurisdiction=AU)

STEP 4 — Layer 2: participant obtains a credential and generates a ZK proof
Issued VC fc0bd3de-5f2a-4b7c-8681-b37dc2c32498 to Alice, signed by issuer did:key:z6Mkq2FBvARHbekt...


Generated ZK proof that the issuer-certified jurisdiction is in the allowed set, without revealing which one.

STEP 5 — Layer 1: AI agents evaluate the transaction, each issuing a signed CRR


Eligibility CRR: outcome=True
   - Jurisdiction check passed: jurisdiction proof bound to an issuer-certified credential and verified.
   - Accreditation credential signature/validity check: passed; issuer authorized: True; issued to sender: True; claimed accredited=True.
Risk-scoring CRR: outcome=0.052
   - Calibrated risk score: 0.052 (0=low risk, 1=high risk).
   - Feature 'amount' = 250000.0 increased the risk score by 0.624 (SHAP attribution).
   - Feature 'tx_velocity_24h' = 1 decreased the risk score by 0.319 (SHAP attribution).
   - Feature 'jurisdiction_tier' = 0 decreased the risk score by 0.312 (SHAP attribution).
Anomaly-detection CRR: outcome (flagged)=False
   - Graph-based anomaly score: 0.500 (severity=low).
   - Short return path detected (length 2): funds sent by this participant return to it within a small number of hops, a pattern associated with layering.

STEP 6 — Layer 3: Transaction Validation Module — final enforcement decision


Transaction TX-DEMO-001 ACCEPTED: True
   - [jurisdiction-AU-real_estate-transfer:max_transaction_amount] amount=250000.0 vs max=10000000 -> PASS
   - [jurisdiction-allowlist-real_estate-transfer:require_jurisdiction_allowlist_proof] ZK jurisdiction-allowlist proof valid (jurisdiction proof bound to an issuer-certified credential and verified) -> PASS
   - [require-eligibility-real_estate-transfer:require_eligibility_crr] eligibility CRR outcome=True -> PASS
   - [max-risk-score-real_estate-transfer:max_risk_score] risk_score=0.052349127446877176 vs max=0.6 -> PASS
   - [block-anomaly-real_estate-transfer:block_if_anomaly_flagged] anomaly_flagged=False -> PASS

STEP 7 — Layer 3: Audit Log integrity check
Audit log length: 68 entries
Audit log integrity intact: True


## Inspecting the audit trail

Every stage across all four layers wrote to the same hash-chained ledger.

In [2]:
audit_log = outcome['env']['audit_log']
print('Total entries:', len(audit_log))
print('Chain integrity intact:', audit_log.verify_integrity())
print()

import collections
counts = collections.Counter(e.event_type for e in audit_log.entries())
for event_type, n in counts.most_common():
    print(f'  {event_type:<35} {n}')

Total entries: 68
Chain integrity intact: True

  governance.vote_cast                30
  governance.finalized                10
  governance.role_proposal_created    5
  role_registry.update_applied        5
  governance.proposal_created         5
  rule_registry.update_applied        5
  governance.member_added             3
  governance.oracle_provider_authorized 3
  asset_registry.registered           1
  transaction.validated               1


## Tamper-evidence, demonstrated

Directly mutating a past entry's payload — exactly what an attacker (or a storage bug) trying to rewrite compliance history would attempt — is detected.

In [3]:
print('Before tamper:', audit_log.verify_integrity())
audit_log._entries[3].payload['amount'] = 999999999
print('After tampering with entry #3:', audit_log.verify_integrity())

Before tamper: True
After tampering with entry #3: False


## The on-chain runtime enforcement layer

If the Solidity contracts have been compiled (`npm install && node scripts/compile_contracts.js`), the cell below deploys them to a local, in-memory EVM and shows transfer-hook enforcement at the contract level, the same mechanism token-level standards such as ERC-3643 use: a transfer to a non-allow-listed address reverts, whichever way the transaction was constructed. This demonstrates the interface, not chain-level enforcement.

In [4]:
build_path = REPO_ROOT / 'arca' / 'layer3_runtime' / 'contracts' / 'build' / 'contracts.json'
if build_path.exists():
    from arca.layer3_runtime.evm_bridge import LocalEVM

    evm = LocalEVM()
    owner, alice, mallory = evm.owner, evm.accounts[1], evm.accounts[3]
    evm.set_allowed(owner, True)
    evm.set_allowed(alice, True)  # mallory deliberately left un-allow-listed

    ok, _ = evm.transfer(owner, alice, 1000)
    print('owner -> alice (both compliant):', ok)

    ok2, err2 = evm.transfer(alice, mallory, 500)
    print('alice -> mallory (bypass attempt):', ok2, '| revert reason:', (err2 or '').splitlines()[0] if err2 else None)
else:
    print('Contracts not compiled yet — run: npm install && node scripts/compile_contracts.js')

owner -> alice (both compliant): True
alice -> mallory (bypass attempt): False | revert reason: execution reverted: ARCA: receiver not compliant


## Threat model

The four scenarios above (and Threat Classes II–IV, which need a dedicated setup) are formalized as executable tests in `tests/test_threat_model.py`, mapped 1:1 onto paper Sec. VI-B, with further trust-check tests in `tests/test_trust_checks.py`. Run them with:

```bash
pytest tests/test_threat_model.py tests/test_trust_checks.py -v
```